# Notebook 07 - Laboratorio Final

## Objetivos
- Resolver un caso empresarial completo con prompt engineering.
- Integrar tecnicas: role + context + few-shot + restricciones.
- Evaluar y documentar el sistema de prompts.

## Introduccion
Caso: **TechNova SaaS** necesita un asistente de soporte que clasifique tickets, responda con base en documentacion y genere resumen ejecutivo.

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path('..')
DATASETS = BASE / 'datasets'
print('Entorno listo. Datasets:', list(DATASETS.glob('*.csv')))

In [ ]:
from transformers import pipeline, set_seed

set_seed(42)
generator = pipeline('text-generation', model='datificate/gpt2-small-spanish')
print('GPT-2 en español listo para experimentos de prompting')

In [ ]:
def generar(prompt, max_new_tokens=50, temperature=0.7):
    out = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    return out[0]['generated_text']

print('Funcion generar() lista')

## 1) Contexto del caso

In [ ]:
df_casos = pd.read_csv(DATASETS / 'casos_practicos.csv')
caso = df_casos[df_casos['empresa'] == 'TechNova SaaS'].iloc[0]
display(pd.DataFrame([caso]))

df_docs = pd.read_csv(DATASETS / 'documentos_empresa.csv')
df_tickets = pd.read_csv(DATASETS / 'tickets_soporte.csv')
print(f'Documentos: {len(df_docs)} | Tickets: {len(df_tickets)}')

## 2) Paso 1: Clasificar tickets con few-shot

In [ ]:
def clasificar_ticket(ticket, ejemplos_df):
    lineas = ['Eres analista de soporte. Clasifica urgencia: alta, media, baja. Solo etiqueta.', '']
    for _, r in ejemplos_df.head(4).iterrows():
        lineas += [f'Ticket: {r["ticket"]}', f'Urgencia: {r["urgencia"]}', '']
    lineas += [f'Ticket: {ticket}', 'Urgencia:']
    return generar('\n'.join(lineas), max_new_tokens=5, temperature=0.3)

clasificaciones = []
for _, row in df_tickets.head(5).iterrows():
    resp = clasificar_ticket(row['ticket'], df_tickets)
    clasificaciones.append({
        'ticket': row['ticket'][:45],
        'urgencia_real': row['urgencia'],
        'prediccion': resp[-15:].strip(),
    })
display(pd.DataFrame(clasificaciones))

## 3) Paso 2: Responder con context prompting

In [ ]:
def responder_con_contexto(pregunta, docs_df):
    contexto = '\n'.join([f'{r["titulo"]}: {r["contenido"]}' for _, r in docs_df.iterrows()])
    prompt = (
        'Eres asistente de soporte de TechNova. Responde SOLO con el contexto. '
        'Si no encuentras la respuesta di No disponible. Maximo 2 oraciones.\n'
        f'<contexto>\n{contexto}\n</contexto>\n'
        f'Cliente: {pregunta}\n'
        'Asistente:'
    )
    return generar(prompt, max_new_tokens=40, temperature=0.3)

preguntas = [
    '¿Puedo devolver un producto después de 25 días?',
    '¿Cuántos usuarios incluye el plan Enterprise?',
    '¿Cuál es el salario del CEO?',
]
respuestas = []
for p in preguntas:
    r = responder_con_contexto(p, df_docs)
    respuestas.append({'pregunta': p, 'respuesta': r[-80:]})
display(pd.DataFrame(respuestas))

## 4) Paso 3: Resumen ejecutivo con role prompting

In [ ]:
tickets_urgentes = df_tickets[df_tickets['urgencia'] == 'alta']['ticket'].tolist()
texto_tickets = '\n'.join(f'- {t}' for t in tickets_urgentes)

prompt_resumen = (
    'Eres gerente de soporte redactando un resumen ejecutivo.\n'
    'Resume los tickets urgentes en 3 viñetas. Tono profesional.\n'
    f'Tickets urgentes:\n{texto_tickets}\n'
    'Resumen ejecutivo:'
)
print(generar(prompt_resumen, max_new_tokens=60, temperature=0.5))

## 5) Paso 4: Sistema integrado

In [ ]:
def asistente_techNova(ticket_o_pregunta, modo='clasificar'):
    if modo == 'clasificar':
        return clasificar_ticket(ticket_o_pregunta, df_tickets)
    elif modo == 'responder':
        return responder_con_contexto(ticket_o_pregunta, df_docs)
    elif modo == 'resumir':
        return generar(
            f'Resume en 2 viñetas: {ticket_o_pregunta}\nResumen:',
            max_new_tokens=40, temperature=0.4,
        )
    return 'Modo no valido'

demo = pd.DataFrame([
    {'modo': 'clasificar', 'entrada': 'La app se cierra al abrir PDFs', 'salida': asistente_techNova('La app se cierra al abrir PDFs', 'clasificar')[-20:]},
    {'modo': 'responder', 'entrada': '¿Cuál es la política de devoluciones?', 'salida': asistente_techNova('¿Cuál es la política de devoluciones?', 'responder')[-60:]},
    {'modo': 'resumir', 'entrada': 'Varios fallos de inicio de sesion reportados', 'salida': asistente_techNova('Varios fallos de inicio de sesion reportados', 'resumir')[-50:]},
])
display(demo)

## 6) Documentacion del sistema de prompts

In [ ]:
documentacion = pd.DataFrame([
    {'funcion': 'clasificar_ticket', 'tecnica': 'few-shot', 'restricciones': 'Solo etiqueta', 'temperatura': 0.3},
    {'funcion': 'responder_con_contexto', 'tecnica': 'context', 'restricciones': 'Solo contexto + No disponible', 'temperatura': 0.3},
    {'funcion': 'resumen ejecutivo', 'tecnica': 'role', 'restricciones': '3 viñetas, tono profesional', 'temperatura': 0.5},
])
display(documentacion)

## Resultados
Construimos un asistente de soporte para TechNova con 3 funciones: clasificacion (few-shot), respuestas (context) y resumen (role).

## Conclusiones
Un sistema real combina multiples tecnicas. La documentacion de prompts es tan importante como el codigo para mantenimiento y escalabilidad.

## Ejercicios guiados resueltos
**Ejercicio:** Agrega una funcion de traduccion al asistente.

**Solucion:**

In [ ]:
def traducir(texto):
    p = f'Parafrasea en español formal. Solo el texto:\nOriginal: {texto}\nParafraseo:'
    return generar(p, max_new_tokens=20, temperature=0.3)
print(traducir('No puedo acceder al sistema'))

## Ejercicios propuestos
1. Mejora accuracy de clasificacion con mas ejemplos.
2. Agrega evaluacion automatica con keywords.
3. Diseña el asistente para otro caso del CSV.

## Preguntas de reflexion
1. Que funcion fue mas dificil de promptear?
2. Como desplegarias esto con una API real (OpenAI)?
3. Que haria falta para pasar de GPT-2 español a produccion?